# NLP 课堂实践：构建React Agent系统

**实践内容概述**: 部署React Agent系统, 首先完成一个简单联网查询的小例子，掌握React的基本迭代思想。然后以Hotpot QA作为实验数据集, 进一步探究React Agent的核心思想

**实践目录**：
1. **React基础知识**：掌握React Agent基本思想和结构
2. **React Agent环境配置与初步代码实现**：掌握React Agent环境部署、配置大模型API、动手实践基本的react联网查询应用。
3. **React Agent测评**: 在第二步基础上, 下载Hotpot数据集来 QA和EM、F1指标来评测React Agent能力
4. **实验测试**：编写实验代码，完成React Agent构建及评估框架
5. **量化评估**：根据实验结果，探究直接调用LLM 和 利用React Agent完成QA的效果
6. **课后作业**：通过三道课后作业，自行设计新的tool工具，进一步加深对Agent理解

## React Agent 基础知识
Agent作为能够感知环境、做出决策并采取行动以实现特定目标的自主实体。与传统直接调用语言模型相比，Agent 具备以下核心特征：
* 自主性：无需人工干预即可独立运行
* 反应性：能对环境变化做出实时响应
* 主动性：主动追求目标而非被动响应
* 社会性：像人类进行交互

React Agent = Reasoning + Acting， 即不让LLM一次性回答出问题的最终答案，而是让它不断地与外界真实环境和工具进行交互，通过多次“思考”和“行动”，根据环境反馈，再进行下一步决策

React每一轮迭代LLM输出结构：

(1) Thought: 思考为什么要这么做（LLM 内部推理）

(2) Action: 调用哪些 Tool（函数名，例如 "search"）

(3) Observation: 工具返回运行结果

(4) Loop： Agent根据观察到的结果继续思考
 

ReAct是一个常用的基础框架，相对于Plan-and-Solve等Agent框架，其复杂度低，控制性强
![图片说明](image.png)



## React Agent环境配置
1.配置python虚拟环境
```python
conda create --name codeAgent python==3.11 -y
conda activate codeAgent
```
2.安装环境
```python
pip install -r requirement.txt
```
3.配置LLM API
```shell
# .env
LLM_API_KEY=""
LLM_MODEL_ID="glm-4.7-flash"
LLM_BASE_URL="https://open.bigmodel.cn/api/paas/v4"
```

## React Agent环境配置与初步代码实现
1. 检查大模型可访问性
python -m agent.llm
```python
if __name__ == "__main__":
    llm = HelloAgentsLLM()
    reply = llm.think([
        {"role": "user", "content": "请用一句话自我介绍。"},
    ])
    print("\n--- 测试结果 ---")
    print("通畅" if reply else "失败")
```

2.编写类，实现基本ReactAgent

```python   
class ReActAgent:
    def __init__(self, llm_client: HelloAgentsLLM, tool_executor: ToolExecutor, max_steps: int = 5):
        self.llm_client = llm_client
        self.tool_executor = tool_executor
        self.max_steps = max_steps
        self.history = []

    def run(self, question: str):
        """
        运行ReAct智能体来回答一个问题。
        """
        self.history = [] # 每次运行时重置历史记录
        current_step = 0

        while current_step < self.max_steps:
            current_step += 1
            print(f"--- 第 {current_step} 步 ---")

            # 1. 格式化提示词
            tools_desc = self.tool_executor.getAvailableTools()
            history_str = "\n".join(self.history)
            prompt = REACT_PROMPT_TEMPLATE.format(
                tools=tools_desc,
                question=question,
                history=history_str
            )

            # 2. 调用LLM进行思考
            messages = [{"role": "user", "content": prompt}]
            response_text = self.llm_client.think(messages=messages)
            
            if not response_text:
                print("错误:LLM未能返回有效响应。")
                break
            # (这段逻辑在 run 方法的 while 循环内)
            # 3. 解析LLM的输出
            thought, action = self._parse_output(response_text)
            
            if thought:
                print(f"思考: {thought}")

            if not action:
                print("警告:未能解析出有效的Action，流程终止。")
                break

            # 4. 执行Action
            if action.startswith("Finish"):
                # 如果是Finish指令，提取最终答案并结束
                m = re.match(r"Finish\[(.*)\]", action, re.DOTALL)
                final_answer = m.group(1).strip() if m else action[len("Finish"):].strip(" :[]\n")
                print(f" 最终答案: {final_answer}")
                return final_answer
            
            tool_name, tool_input = self._parse_action(action)
            if not tool_name or not tool_input:
                # ... 处理无效Action格式 ...
                continue

            print(f" 行动: {tool_name}[{tool_input}]")
            
            tool_function = self.tool_executor.getTool(tool_name)
            if not tool_function:
                observation = f"错误:未找到名为 '{tool_name}' 的工具。"
            else:
                observation = tool_function(tool_input) # 调用真实工具
            print(f" 观察: {observation}")
            
            # 将本轮的Action和Observation添加到历史记录中
            self.history.append(f"Action: {action}")
            self.history.append(f"Observation: {observation}")

        # 循环结束:强制让LLM根据已有历史给出最终答案
        print(f"已达到最大步数({self.max_steps})，强制输出最终答案。")
        history_str = "\n".join(self.history)
        force_prompt = (
            f"你已达到最大思考步数,必须立刻给出最终答案,不能再调用任何工具。\n"
            f"请基于以下历史信息直接回答用户的问题。\n\n"
            f"Question: {question}\n"
            f"History:\n{history_str}\n\n"
            f"请只输出最终答案文本,不要再输出 Thought / Action / Finish 等标记。"
        )
        final_answer = self.llm_client.think(
            messages=[{"role": "user", "content": force_prompt}]
        )
        if final_answer:
            final_answer = final_answer.strip()
            print(f" 最终答案: {final_answer}")
        else:
            print(" 警告:LLM未能给出最终答案。")
        return final_answer

    def _parse_output(self, text: str):
        """解析LLM的输出，提取Thought和Action。
        """
        # Thought: 匹配到 Action: 或文本末尾
        thought_match = re.search(r"Thought:\s*(.*?)(?=\nAction:|$)", text, re.DOTALL)
        # Action: 匹配到文本末尾
        action_match = re.search(r"Action:\s*(.*?)$", text, re.DOTALL)
        thought = thought_match.group(1).strip() if thought_match else None
        action = action_match.group(1).strip() if action_match else None
        return thought, action

    def _parse_action(self, action_text: str):
        """解析Action字符串，提取工具名称和输入。
        """
        match = re.match(r"(\w+)\[(.*)\]", action_text, re.DOTALL)
        if match:
            return match.group(1), match.group(2)
        return None, None
```
3.编写search工具，用于agent联网搜索
```python
def search(query: str) -> str:
    """
    一个基于SerpApi的实战网页搜索引擎工具。
    它会智能地解析搜索结果，优先返回直接答案或知识图谱信息。
    """
    print(f" 正在执行 [SerpApi] 网页搜索: {query}")
    try:
        api_key = os.getenv("SERPAPI_API_KEY")
        if not api_key:
            return "错误：SERPAPI_API_KEY 未在 .env 文件中配置。"

        params = {
            "engine": "google",
            "q": query,
            "api_key": api_key,
            "gl": "cn",  # 国家代码
            "hl": "zh-cn", # 语言代码
        }
        
        client = SerpApiClient(params)
        results = client.get_dict()
        
        # 智能解析：优先寻找最直接的答案
        if "answer_box_list" in results:
            return "\n".join(results["answer_box_list"])
        if "answer_box" in results and "answer" in results["answer_box"]:
            return results["answer_box"]["answer"]
        if "knowledge_graph" in results and "description" in results["knowledge_graph"]:
            return results["knowledge_graph"]["description"]
        if "organic_results" in results and results["organic_results"]:
            # 如果没有直接答案，则返回前三个有机结果的摘要
            snippets = [
                f"[{i+1}] {res.get('title', '')}\n{res.get('snippet', '')}"
                for i, res in enumerate(results["organic_results"][:3])
            ]
            return "\n\n".join(snippets)
        
        return f"对不起，没有找到关于 '{query}' 的信息。"

    except Exception as e:
        return f"搜索时发生错误: {e}"
    
from typing import Dict, Any

class ToolExecutor:
    """
    一个工具执行器，负责管理和执行工具。
    """
    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def registerTool(self, name: str, description: str, func: callable):
        """
        向工具箱中注册一个新工具。
        """
        if name in self.tools:
            print(f"警告：工具 '{name}' 已存在，将被覆盖。")
        
        self.tools[name] = {"description": description, "func": func}
        print(f"工具 '{name}' 已注册。")

    def getTool(self, name: str) -> callable:
        """
        根据名称获取一个工具的执行函数。
        """
        return self.tools.get(name, {}).get("func")

    def getAvailableTools(self) -> str:
        """
        获取所有可用工具的格式化描述字符串。
        """
        return "\n".join([
            f"- {name}: {info['description']}" 
            for name, info in self.tools.items()
        ])
```
4.设置prompt
```python
# ReAct 提示词模板
REACT_PROMPT_TEMPLATE = """
请注意，你是一个有能力调用外部工具的智能助手。

可用工具如下:
{tools}

请严格按照以下格式进行回应:

Thought: 你的思考过程，用于分析问题、拆解任务和规划下一步行动。
Action: 你决定采取的行动，必须是以下格式之一:
- `{{tool_name}}[{{tool_input}}]`:调用一个可用工具。
- `Finish[最终答案]`:当你认为已经获得最终答案时。
- 当你收集到足够的信息，能够回答用户的最终问题时，你必须在Action:字段后使用 Finish[最终答案] 来输出最终答案。

现在，请开始解决以下问题:
Question: {question}
History: {history}
"""
```

5.最后编写主程序，测试效果
```python
if __name__ == '__main__':
    llm = HelloAgentsLLM()
    tool_executor = ToolExecutor()
    search_desc = "一个网页搜索引擎。当你需要回答关于时事、事实以及在你的知识库中找不到的信息时，应使用此工具。"
    tool_executor.registerTool("Search", search_desc, search)
    agent = ReActAgent(llm_client=llm, tool_executor=tool_executor)
    question = "华为在2026年发布了哪些手机？他们的主要卖点是什么？"
    agent.run(question)
```

期望结果：
```shell
工具 'Search' 已注册。
--- 第 1 步 ---
 正在调用 zai-org/GLM-4.5-Air 模型...
 大语言模型响应成功:
Thought: 用户询问华为在2026年发布的手机及其主要卖点。由于我的知识库截止于2023年，2026年的信息不在其中，因此我需要使用Search工具来获取最新信息。首先，我将搜索华为在2026年发布的手机型号。如果搜索结果显示相关信息，我会进一步提取每个手机的主要卖点；如果没有，我会尝试调整搜索策略或检查是否有年份错误（例如，用户可能意指2023年或2024年）。搜索将使用中文关键词以匹配用户的问题。

Action: `Search[华为 2026年 发布 手机]`
思考: 用户询问华为在2026年发布的手机及其主要卖点。由于我的知识库截止于2023年，2026年的信息不在其中，因此我需要使用Search工具来获取最新信息。首先，我将搜索华为在2026年发布的手机型号。如果搜索结果显示相关信息，我会进一步提取每个手机的主要卖点；如果没有，我会尝试调整搜索策略或检查是否有年份错误（例如，用户可能意指2023年或2024年）。搜索将使用中文关键词以匹配用户的问题。
--- 第 2 步 ---
 正在调用 zai-org/GLM-4.5-Air 模型...
 大语言模型响应成功:
 ...
--- 第 3 步 ---
...
--- 第 4 步 ---
...
--- 第 5 步 ---
...
已达到最大步数(5)，强制输出最终答案。
 正在调用 zai-org/GLM-4.5-Air 模型...
 大语言模型响应成功:
根据搜索结果，华为在2026年计划发布以下手机及其主要卖点：

1. 华为三折叠手机（Mate XT非凡大师系列）- 计划于9月4日发布
   - 主要卖点：超纤薄三折叠屏设计、10.2英寸大屏、可变光圈主摄、一机多能、适配多种应用场景、主打商务风格
   - 设计特点：传奇星钻设计、正弦切面、岩脉纹理（每片纹理均独一无二）

2. 华为Pura系列 - 计划于4月发布
   - Pura 90 Pro和Pura 90 Pro Max：起售价分别为5499元和6499元
   - Pura X2（第二代阔折叠/竖折）：时尚便携的竖向折叠屏手机，亮点包括铰链技术、外屏交互以及轻薄度的改进

3. 华为Mate全能旗舰系列 - 计划于10月发布
   - 主要卖点：全能配置，涵盖芯片、影像、系统以及通讯等多个核心方面的全面升级

这些信息基于华为官方发布规划和官网信息整理。
 最终答案: 根据搜索结果，华为在2026年计划发布以下手机及其主要卖点：

1. 华为三折叠手机（Mate XT非凡大师系列）- 计划于9月4日发布
   - 主要卖点：超纤薄三折叠屏设计、10.2英寸大屏、可变光圈主摄、一机多能、适配多种应用场景、主打商务风格
   - 设计特点：传奇星钻设计、正弦切面、岩脉纹理（每片纹理均独一无二）

2. 华为Pura系列 - 计划于4月发布
   - Pura 90 Pro和Pura 90 Pro Max：起售价分别为5499元和6499元
   - Pura X2（第二代阔折叠/竖折）：时尚便携的竖向折叠屏手机，亮点包括铰链技术、外屏交互以及轻薄度的改进

3. 华为Mate全能旗舰系列 - 计划于10月发布
   - 主要卖点：全能配置，涵盖芯片、影像、系统以及通讯等多个核心方面的全面升级

这些信息基于华为官方发布规划和官网信息整理。
```


## React Agent测评

### 思路
使用react agent通过动态思考，并从wiki中search与问题相关的内容，回答hotpot QA中的问题。
测评指标如下：

(1)EM (Exact Match)
对预测答案和标准答案做归一化（小写、去标点、去冠词 a/an/the、合并空白）后，完全一致记 1 分，否则 0 分。是一个非常严格的二元指标 —— 多一个字、少一个字都算 0。


(2)F1 (Token-level F1)
把归一化后的预测和标准答案各切成 token 多重集，统计两者重叠的 token 数（按出现次数取较小值，即多重集交集大小），由此算 precision 和 recall，再取调和平均。
允许部分重叠，比 EM 宽松：例如 gold = "Barack Obama"，pred = "Obama" → EM=0，F1≈0.67。




### 实验设置
1.下载hotpot数据集
```python
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "data")
MS_CACHE = os.path.join(DATA_DIR, "ms_cache")
SAMPLE_PATH = os.path.join(DATA_DIR, "hotpot_sample_100.json")


def _ensure_parquet() -> str:
    """从 ModelScope 下载（带缓存）并返回 validation parquet 路径。"""
    expected = os.path.join(
        MS_CACHE, "cmriat", "hotpotqa", "data", "validation-00000-of-00001.parquet"
    )
    if os.path.exists(expected):
        return expected

    from modelscope import snapshot_download

    print("[load_data] downloading cmriat/hotpotqa from ModelScope ...")
    snapshot_download(
        "cmriat/hotpotqa", repo_type="dataset", cache_dir=MS_CACHE
    )
    return expected


def sample(n: int = 100, seed: int = 42) -> List[Dict]:
    if os.path.exists(SAMPLE_PATH):
        with open(SAMPLE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    import pandas as pd

    pq = _ensure_parquet()
    df = pd.read_parquet(pq)
    print(f"[load_data] loaded {len(df)} validation examples")

    rng = random.Random(seed)
    idx = rng.sample(range(len(df)), n)

    sub = []
    for i in idx:
        row = df.iloc[i]
        golds = list(row["golden_answers"])
        meta = row["metadata"] if isinstance(row["metadata"], dict) else {}
        sub.append(
            {
                "_id": row["id"],
                "question": row["question"],
                "answer": golds[0] if golds else "",
                "golden_answers": golds,
                "type": meta.get("type"),
                "level": meta.get("level"),
            }
        )

    os.makedirs(DATA_DIR, exist_ok=True)
    with open(SAMPLE_PATH, "w", encoding="utf-8") as f:
        json.dump(sub, f, ensure_ascii=False, indent=2)
    print(f"[load_data] sampled {n} -> {SAMPLE_PATH}")
    return sub


if __name__ == "__main__":
    n = int(sys.argv[1]) if len(sys.argv) > 1 else 100
    items = sample(n)
    print(f"got {len(items)} items; first:")
    print(json.dumps(items[0], indent=2, ensure_ascii=False))
```
2.编写测评指标
```python
import re
import string
from collections import Counter


def normalize_answer(s: str) -> str:
    if s is None:
        return ""

    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def em_score(pred: str, gold: str) -> int:
    return int(normalize_answer(pred) == normalize_answer(gold))


def f1_score(pred: str, gold: str) -> float:
    pred_toks = normalize_answer(pred).split()
    gold_toks = normalize_answer(gold).split()
    if not pred_toks or not gold_toks:
        return float(pred_toks == gold_toks)
    common = Counter(pred_toks) & Counter(gold_toks)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0
    p = num_same / len(pred_toks)
    r = num_same / len(gold_toks)
    return 2 * p * r / (p + r)
```
3.编写react
```python
class HotpotLLM(HelloAgentsLLM):
    """覆盖 think()：非流式 + 自定义 stop，避免一次吐出多个 step。
    内置最小调用间隔节流（默认 1.2s/次）和 429 指数退避重试。
    """

    def __init__(
        self,
        min_interval: float = 1.2,
        max_retries: int = 6,
        empty_retries: int = 3,
        empty_retry_wait: float = 3.0,
        **kw,
    ):
        super().__init__(**kw)
        self.min_interval = float(os.getenv("LLM_MIN_INTERVAL", min_interval))
        self.max_retries = max_retries
        self.empty_retries = empty_retries
        self.empty_retry_wait = empty_retry_wait
        self._last_call_ts: float = 0.0

    def _throttle(self):
        wait = self.min_interval - (time.time() - self._last_call_ts)
        if wait > 0:
            time.sleep(wait)

    def _call_once(self, prompt: str, temperature: float) -> Optional[str]:
        """单次调用 + 429 指数退避；不处理空返回。"""
        backoff = max(self.min_interval, 2.0)
        for attempt in range(self.max_retries + 1):
            self._throttle()
            self._last_call_ts = time.time()
            try:
                response = self.client.chat.completions.create(
                    model=self.model,
                    messages=[{"role": "user", "content": prompt}],
                    temperature=temperature,
                    stream=False,
                    stop=["\nObservation"],
                )
                return response.choices[0].message.content
            except Exception as e:
                msg = str(e)
                is_rate = (
                    "429" in msg
                    or "1302" in msg
                    or "rate" in msg.lower()
                    or "速率" in msg
                )
                if is_rate and attempt < self.max_retries:
                    print(
                        f"[LLM rate-limited] retry in {backoff:.1f}s (attempt {attempt + 1}/{self.max_retries})"
                    )
                    time.sleep(backoff)
                    backoff = min(backoff * 2, 30.0)
                    continue
                print(f"[LLM error] {e}")
                return None
        return None

    def think_once(self, prompt: str, temperature: float = 0.0) -> Optional[str]:
        """带"空响应"重试：最多 empty_retries 次，每次等 empty_retry_wait 秒。
        每次重试都用稍微提高的温度，避免确定性贪心反复落到空输出。"""
        for attempt in range(self.empty_retries + 1):
            t = temperature + 0.2 * attempt
            resp = self._call_once(prompt, temperature=t)
            if resp and resp.strip():
                return resp
            if attempt < self.empty_retries:
                print(
                    f"[LLM empty] retry {attempt + 1}/{self.empty_retries} in {self.empty_retry_wait:.1f}s (temp={t + 0.2:.1f})"
                )
                time.sleep(self.empty_retry_wait)
        return None


class HotpotReActAgent:
    def __init__(self, llm: HotpotLLM, max_steps: int = 7, verbose: bool = False):
        self.llm = llm
        self.max_steps = max_steps
        self.verbose = verbose
        self.wiki = WikiEnv()

    def _build_prompt(self, question: str, history: str, next_step: int) -> str:
        # 在末尾显式给出下一步的 "Thought N:" 前缀，避免模型续写出 \nObservation 触发 stop。
        tail = f"\nThought {next_step}:"
        return (
            HOTPOT_REACT_PROMPT.replace("{question}", question).replace(
                "{history}", history
            )
            + tail
        )

    @staticmethod
    def _parse_step(text: str, step_i: int) -> Tuple[Optional[str], Optional[str]]:
        # 因为 prompt 末尾已经给了 "Thought N:"，模型直接续写 thought 内容，再换行写 "Action N:"。
        # 优先按这种"裸续写"格式解析；解析不到再回退到带 "Thought N:" 前缀的格式。
        bare_t = re.match(r"\s*(.*?)(?=\n\s*Action\b|\Z)", text, re.DOTALL)
        bare_a = re.search(
            rf"Action\s*{step_i}?\s*:\s*(.+?)(?:\n|$)", text, re.DOTALL
        )

        thought = bare_t.group(1).strip() if bare_t else None
        action = bare_a.group(1).strip() if bare_a else None

        # 回退：如果模型自己又写了 "Thought N:"（重复 prompt 末尾），重新提取
        t2 = re.search(
            rf"Thought\s*{step_i}\s*:\s*(.*?)(?=\n\s*Action\s*{step_i}\s*:|$)",
            text,
            re.DOTALL,
        )
        if t2:
            thought = t2.group(1).strip()
        if not action:
            a2 = re.search(r"Action[^:\n]*:\s*(.+?)(?:\n|$)", text, re.DOTALL)
            if a2:
                action = a2.group(1).strip()
        return thought, action

    @staticmethod
    def _parse_action(action: str) -> Tuple[Optional[str], Optional[str]]:
        m = re.match(r"(\w+)\s*\[(.*)\]\s*$", action, re.DOTALL)
        if m:
            return m.group(1).strip(), m.group(2).strip()
        return None, None

    def run(self, question: str) -> dict:
        self.wiki.reset()
        history = ""
        trace = []
        final_answer = None

        for step in range(1, self.max_steps + 1):
            prompt = self._build_prompt(question, history, step)
            response = self.llm.think_once(prompt)
            if not response:
                print(f"[step {step}] empty LLM response, abort")
                break

            thought, action = self._parse_step(response, step)
            if self.verbose:
                print(f"[step {step}] thought={thought!r} action={action!r}")

            if not action:
                print(f"[step {step}] no action parsed; raw={response!r}")
                trace.append({"step": step, "raw": response})
                break

            trace.append({"step": step, "thought": thought, "action": action})

            tool, arg = self._parse_action(action)
            if tool is None:
                observation = (
                    "Invalid action. Use Search[...], Lookup[...], or Finish[...]."
                )
            elif tool.lower() == "finish":
                final_answer = arg
                trace[-1]["observation"] = arg
                break
            elif tool.lower() == "search":
                observation = self.wiki.search(arg)
            elif tool.lower() == "lookup":
                observation = self.wiki.lookup(arg)
            else:
                observation = (
                    f"Invalid action {tool}. Use Search/Lookup/Finish."
                )

            trace[-1]["observation"] = observation
            if self.verbose:
                obs_show = observation if len(observation) < 200 else observation[:200] + "..."
                print(f"[step {step}] obs={obs_show}")

            history += (
                f"\nThought {step}: {thought or ''}"
                f"\nAction {step}: {action}"
                f"\nObservation {step}: {observation}"
            )

        return {
            "question": question,
            "answer": final_answer,
            "steps": len(trace),
            "trace": trace,
        }


if __name__ == "__main__":
    agent = HotpotReActAgent(HotpotLLM(), verbose=True)
    out = agent.run(
        "What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?"
    )
    print("FINAL:", out["answer"])
```
4.编写wiki search函数
```python
import re
import requests
from typing import List, Optional

WIKI_API = "https://en.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "ReActBenchmark/1.0 (research)"}


def _split_sentences(text: str) -> List[str]:
    text = re.sub(r"\s+", " ", text).strip()
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9\"'(])", text)
    return [p.strip() for p in parts if p.strip()]


def _clean_wiki_text(text: str) -> str:
    text = re.sub(r"==+.*?==+", " ", text)
    text = re.sub(r"\{\{.*?\}\}", " ", text, flags=re.DOTALL)
    text = re.sub(r"<.*?>", " ", text, flags=re.DOTALL)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


class WikiEnv:
    """维护当前页面状态（供 lookup 使用）。"""

    def __init__(self, timeout: int = 30, retries: int = 2):
        self.timeout = timeout
        self.retries = retries
        self.page_title: Optional[str] = None
        self.page_sentences: List[str] = []
        self.last_lookup: Optional[str] = None
        self.lookup_results: List[int] = []
        self.lookup_idx: int = 0

    def _get_json(self, params: dict) -> Optional[dict]:
        last = None
        for attempt in range(self.retries + 1):
            try:
                r = requests.get(
                    WIKI_API, params=params, headers=HEADERS, timeout=self.timeout
                )
                r.raise_for_status()
                return r.json()
            except Exception as e:
                last = e
        print(f"[wiki_tools] request failed after retries: {last}")
        return None

    def _get_page_extract(self, title: str) -> Optional[str]:
        data = self._get_json(
            {
                "action": "query",
                "format": "json",
                "prop": "extracts",
                "explaintext": 1,
                "redirects": 1,
                "titles": title,
            }
        )
        if not data:
            return None
        pages = data.get("query", {}).get("pages", {})
        for _, p in pages.items():
            if "missing" in p:
                return None
            return p.get("extract")
        return None

    def _search_titles(self, query: str, limit: int = 5) -> List[str]:
        data = self._get_json(
            {
                "action": "query",
                "format": "json",
                "list": "search",
                "srsearch": query,
                "srlimit": limit,
            }
        )
        if not data:
            return []
        hits = data.get("query", {}).get("search", [])
        return [h["title"] for h in hits]

    def search(self, entity: str) -> str:
        entity = entity.strip().strip('"').strip("'")
        if not entity:
            return "Invalid search query."

        extract = self._get_page_extract(entity)
        if extract:
            extract = _clean_wiki_text(extract)
            sents = _split_sentences(extract)
            if sents:
                self.page_title = entity
                self.page_sentences = sents
                self.last_lookup = None
                self.lookup_results = []
                self.lookup_idx = 0
                return " ".join(sents[:5])

        titles = self._search_titles(entity, limit=5)
        if titles:
            return f"Could not find {entity}. Similar: {titles}."
        return f"Could not find {entity}. Similar: []."

    def lookup(self, keyword: str) -> str:
        keyword = keyword.strip().strip('"').strip("'")
        if not self.page_sentences:
            return "No page loaded. Use Search first."

        if keyword.lower() != (self.last_lookup or "").lower():
            self.last_lookup = keyword
            kw = keyword.lower()
            self.lookup_results = [
                i for i, s in enumerate(self.page_sentences) if kw in s.lower()
            ]
            self.lookup_idx = 0

        if not self.lookup_results:
            return f"No results for \"{keyword}\"."

        if self.lookup_idx >= len(self.lookup_results):
            return f"No more results. ({len(self.lookup_results)} total for \"{keyword}\")"

        sent_i = self.lookup_results[self.lookup_idx]
        self.lookup_idx += 1
        return (
            f"(Result {self.lookup_idx} / {len(self.lookup_results)}) "
            + self.page_sentences[sent_i]
        )

    def reset(self):
        self.page_title = None
        self.page_sentences = []
        self.last_lookup = None
        self.lookup_results = []
        self.lookup_idx = 0


if __name__ == "__main__":
    env = WikiEnv()
    print(env.search("Colorado orogeny"))
    print("---")
    print(env.lookup("eastern sector"))
```
5.设置好prompt
```python
"""
HotpotQA ReAct 提示词 —— few-shot 示例改写自论文 Appendix C (Yao et al., 2023)。
"""

HOTPOT_REACT_PROMPT = """Solve a question answering task with interleaving Thought, Action, Observation steps.
Thought can reason about the current situation, and Action can be three types:
(1) Search[entity], which searches the exact entity on Wikipedia and returns the first sentences if it exists. If not, it will return some similar entities to search.
(2) Lookup[keyword], which returns the next sentence containing keyword in the current passage.
(3) Finish[answer], which returns the answer and finishes the task.
You may take as many steps as necessary, but try to be concise.

Here are some examples:

Question: What is the elevation range for the area that the eastern sector of the Colorado orogeny extends into?
Thought 1: I need to search Colorado orogeny, find the area that the eastern sector of the Colorado orogeny extends into, then find the elevation range of the area.
Action 1: Search[Colorado orogeny]
Observation 1: The Colorado orogeny was an episode of mountain building (an orogeny) in Colorado and surrounding areas.
Thought 2: It does not mention the eastern sector. So I need to look up eastern sector.
Action 2: Lookup[eastern sector]
Observation 2: (Result 1 / 1) The eastern sector extends into the High Plains and is called the Central Plains orogeny.
Thought 3: The eastern sector of Colorado orogeny extends into the High Plains. So I need to search High Plains and find its elevation range.
Action 3: Search[High Plains]
Observation 3: High Plains refers to one of two distinct land regions.
Thought 4: I need to instead search High Plains (United States).
Action 4: Search[High Plains (United States)]
Observation 4: The High Plains are a subregion of the Great Plains. From east to west, the High Plains rise in elevation from around 1,800 to 7,000 ft (550 to 2,130 m).
Thought 5: High Plains rise in elevation from around 1,800 to 7,000 ft, so the answer is 1,800 to 7,000 ft.
Action 5: Finish[1,800 to 7,000 ft]

Question: Musician and satirist Allie Goertz wrote a song about the "The Simpsons" character Milhouse, who Matt Groening named after who?
Thought 1: The question simplifies to "The Simpsons" character Milhouse is named after who. I only need to search Milhouse and find who it is named after.
Action 1: Search[Milhouse]
Observation 1: Milhouse Mussolini Van Houten is a recurring character in the Fox animated television series The Simpsons voiced by Pamela Hayden and created by Matt Groening.
Thought 2: The paragraph does not tell who Milhouse is named after, maybe I can look up "named after".
Action 2: Lookup[named after]
Observation 2: (Result 1 / 1) Milhouse was named after U.S. president Richard Nixon, whose middle name was Milhous.
Thought 3: Milhouse was named after U.S. president Richard Nixon, so the answer is Richard Nixon.
Action 3: Finish[Richard Nixon]

Question: Which documentary is about Finnish rock groups, Adam Clayton Powell or The Saimaa Gesture?
Thought 1: I need to search Adam Clayton Powell and The Saimaa Gesture, and find which documentary is about Finnish rock groups.
Action 1: Search[Adam Clayton Powell]
Observation 1: Could not find Adam Clayton Powell. Similar: ['Adam Clayton Powell III', 'Seventh Avenue (Manhattan)', 'Adam Clayton Powell Jr. State Office Building', 'Isabel Washington Powell', 'Adam Powell', 'Adam Clayton Powell (film)', 'Giancarlo Esposito'].
Thought 2: To find the documentary, I can search Adam Clayton Powell (film).
Action 2: Search[Adam Clayton Powell (film)]
Observation 2: Adam Clayton Powell is a 1989 American documentary film directed by Richard Kilberg. The film is about the rise and fall of influential African-American politician Adam Clayton Powell Jr.
Thought 3: Adam Clayton Powell (film) is a documentary about an African-American politician, not Finnish rock groups. So the documentary about Finnish rock groups must instead be The Saimaa Gesture.
Action 3: Finish[The Saimaa Gesture]

Question: What profession does Nicholas Ray and Elia Kazan have in common?
Thought 1: I need to search Nicholas Ray and Elia Kazan, find their professions, then find the profession they have in common.
Action 1: Search[Nicholas Ray]
Observation 1: Nicholas Ray (born Raymond Nicholas Kienzle Jr., August 7, 1911 - June 16, 1979) was an American film director, screenwriter, and actor best known for the 1955 film Rebel Without a Cause.
Thought 2: Professions of Nicholas Ray are director, screenwriter, and actor. I need to search Elia Kazan next and find his professions.
Action 2: Search[Elia Kazan]
Observation 2: Elia Kazan was an American film and theatre director, producer, screenwriter and actor.
Thought 3: Professions of Elia Kazan are director, producer, screenwriter, and actor. So profession Nicholas Ray and Elia Kazan have in common is director, screenwriter, and actor.
Action 3: Finish[director, screenwriter, actor]

Question: Which magazine was started first Arthur's Magazine or First for Women?
Thought 1: I need to search Arthur's Magazine and First for Women, and find which was started first.
Action 1: Search[Arthur's Magazine]
Observation 1: Arthur's Magazine (1844-1846) was an American literary periodical published in Philadelphia in the 19th century.
Thought 2: Arthur's Magazine was started in 1844. I need to search First for Women next.
Action 2: Search[First for Women]
Observation 2: First for Women is a woman's magazine published by Bauer Media Group in the USA. The magazine was started in 1989.
Thought 3: First for Women was started in 1989. 1844 (Arthur's Magazine) < 1989 (First for Women), so Arthur's Magazine was started first.
Action 3: Finish[Arthur's Magazine]

Question: Were Pavel Urysohn and Leonid Levin known for the same type of work?
Thought 1: I need to search Pavel Urysohn and Leonid Levin, find their types of work, then find if they are the same.
Action 1: Search[Pavel Urysohn]
Observation 1: Pavel Samuilovich Urysohn (February 3, 1898 - August 17, 1924) was a Soviet mathematician who is best known for his contributions in dimension theory.
Thought 2: Pavel Urysohn is a mathematician. I need to search Leonid Levin next and find his type of work.
Action 2: Search[Leonid Levin]
Observation 2: Leonid Anatolievich Levin is a Soviet-American mathematician and computer scientist.
Thought 3: Leonid Levin is a mathematician and computer scientist. So Pavel Urysohn and Leonid Levin have the same type of work.
Action 3: Finish[yes]

(End of examples.)

Now answer the following question. Respond ONLY with the next single line in the form:
Thought {i}: <your reasoning>
Action {i}: <one of Search[...], Lookup[...], Finish[...]>

Question: {question}
{history}"""
```

6. 编写测试文件
```python
import argparse
import json
import os
import sys
import time

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from benchmark.evaluate import em_score, f1_score
from benchmark.hotpot_react import HotpotLLM, HotpotReActAgent
from benchmark.load_data import sample

RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath(__file__)), "results")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=100)
    ap.add_argument("--max_steps", type=int, default=7)
    ap.add_argument("--verbose", action="store_true")
    ap.add_argument("--out", type=str, default=None)
    ap.add_argument(
        "--min_interval",
        type=float,
        default=1.2,
        help="LLM 两次请求之间的最小间隔（秒），用于绕过 1 QPS 限速",
    )
    args = ap.parse_args()

    items = sample(args.n)[: args.n]
    print(f"[benchmark] running on {len(items)} examples")

    llm = HotpotLLM(min_interval=args.min_interval)
    agent = HotpotReActAgent(llm, max_steps=args.max_steps, verbose=args.verbose)
    print(f"[benchmark] LLM min_interval={llm.min_interval}s, max_retries={llm.max_retries}")

    os.makedirs(RESULTS_DIR, exist_ok=True)
    out_path = args.out or os.path.join(
        RESULTS_DIR, f"react_hotpot_n{args.n}_{int(time.time())}.json"
    )

    records = []
    em_sum = 0.0
    f1_sum = 0.0
    t0 = time.time()
    for i, ex in enumerate(items):
        q = ex["question"]
        gold = ex["answer"]
        print(f"\n===== [{i + 1}/{len(items)}] {q[:120]}")
        try:
            res = agent.run(q)
        except Exception as e:
            print(f"[error] {e}")
            res = {"answer": None, "steps": 0, "trace": [], "error": str(e)}
        pred = res.get("answer") or ""
        em = em_score(pred, gold)
        f1 = f1_score(pred, gold)
        em_sum += em
        f1_sum += f1
        rec = {
            "_id": ex["_id"],
            "question": q,
            "gold": gold,
            "pred": pred,
            "em": em,
            "f1": f1,
            "steps": res.get("steps"),
            "trace": res.get("trace"),
            "type": ex.get("type"),
            "level": ex.get("level"),
        }
        records.append(rec)
        print(f"  gold={gold!r} | pred={pred!r} | EM={em} F1={f1:.3f}")
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(
                {
                    "n_done": len(records),
                    "em": em_sum / len(records),
                    "f1": f1_sum / len(records),
                    "records": records,
                },
                f,
                ensure_ascii=False,
                indent=2,
            )

    dt = time.time() - t0
    n = len(records)
    print("\n========= SUMMARY =========")
    print(f"N        : {n}")
    print(f"EM       : {em_sum / n:.4f}")
    print(f"F1       : {f1_sum / n:.4f}")
    print(f"elapsed  : {dt:.1f}s ({dt / n:.1f}s/example)")
    print(f"results  : {out_path}")


if __name__ == "__main__":
    main()
```

使用如下命令运行
```shell
python -m benchmark.run_benchmark --n 1     # 跑前 1 条做小测
```


## 量化评估
在执行`python -m benchmark.run_benchmark --n 2 --verbose --min_interval 2`命令后，我们尝试让react回答两个hotpot中的题目，预期结果如下：
```shell
[benchmark] running on 2 examples
[benchmark] LLM min_interval=2.0s, max_retries=6

===== [1/2] In what year was the university where Sergei Aleksandrovich Tokarev was a professor founded?
[step 1] thought='I need to find the university where Sergei Aleksandrovich Tokarev was a professor, then find the year that university was founded.' action='Search[Sergei Aleksandrovich Tokarev]'
[step 1] obs=Sergei Aleksandrovich Tokarev (Russian: Серге́й Алекса́ндрович То́карев; 29 December 1899 – 19 April 1985) was a Russian scholar, ethnographer, historian, researcher of religious beliefs, doctor of hi...
[step 2] thought='Sergei Aleksandrovich Tokarev was a professor at Moscow State University. I need to search Moscow State University to find the year it was founded.' action='Search[Moscow State University]'
[step 2] obs=Moscow State University (MSU), officially M. V. Lomonosov Moscow State University, is a public research university in Moscow, Russia. The university includes 15 research institutes, 43 faculties, more...
[step 3] thought='The search result for Moscow State University does not mention the year it was founded. I need to look up "founded" to find the specific year.' action='Lookup[founded]'
[step 3] obs=(Result 1 / 2) Though Moscow State University was founded in 1755, St.
[step 4] thought='The observation states that Moscow State University was founded in 1755. This answers the question.' action='Finish[1755]'
  gold='1755' | pred='1755' | EM=1 F1=1.000

========= SUMMARY =========
N        : 1
EM       : 1.0000
F1       : 1.0000
elapsed  : 128.0s (128.0s/example)
results  : /xxx/xxx.json
```

# 讨论
1. 大语言模型超参数如温度等指标，是否会对action步骤产生影响？
2. 如何避免agent陷入多轮对话而无法终止？
3. 如果让agent从多次对话中总结经验，使其在下一次对话中节省tokens的使用？
4. 实验过程中，出现大语言模型响应错误的情况时，你是如何解决的？
5. 设计更多的工具，如代码检测、新闻总结等，想一想是否可以将上一节课的RAG系统以工具形式结合到react agent中，增强agent的检索效率和准确率？

# 参考文献
[1] Yao S, Zhao J, Yu D, et al. React: Synergizing reasoning and acting in language models[C]//International Conference on Learning Representations (ICLR). 2023.